# covattn - run experiments on Colab
Runtime -> Change runtime type -> **A100 GPU**. Run cells 1-5 once per session; every experiment cell uses `--resume`, so after a disconnect just re-run 1-5 and the experiment cell.

Scripts run as subprocesses (`!python ...`), so pinned packages take effect without restarting the runtime.

In [ ]:
# 1. mount Drive (results live here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. get / update the code from GitHub
REPO = 'https://github.com/FlyMe2star/covattn.git'
BRANCH = 'main'
import os
if not os.path.isdir('/content/covattn/.git'):
    !git clone -q -b {BRANCH} {REPO} /content/covattn
else:
    !git -C /content/covattn fetch -q origin && git -C /content/covattn reset -q --hard origin/{BRANCH}
%cd /content/covattn
!git log --oneline -3

In [ ]:
# 3. install pinned dependencies (keeps Colab's own CUDA torch)
!pip install -q -r requirements.txt && pip install -q -e . --no-deps
# Colab ships torchao 0.10, which peft 0.21 rejects when it builds LoRA layers; nothing here needs torchao
!pip uninstall -y -q torchao

In [ ]:
# 4. GPU check
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python -c "import torch, chronos; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-', '| chronos', chronos.__version__)"

In [ ]:
# 5. results directory on Drive
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/pakdd27/results'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.makedirs(os.environ['RESULTS_DIR'], exist_ok=True)
print(os.environ['RESULTS_DIR'])

## Smoke test (about 2 min): unit + model tests, source checks, Exp1 quick

In [ ]:
!python -m pytest -q
!python scripts/verify_model.py > /dev/null && echo verify_model OK
!python scripts/exp1_phenomenon.py --quick --resume --run-name colab_quick 2>&1 | grep -E 'ESTIMATED|done|Error|Traceback'

## Stage 0 - Exp1 (phenomenon)
The canonical run `exp1/main` (304 cells) is already on Drive; this cell only checks it.

In [ ]:
!ls $RESULTS_DIR/exp1/main && echo "cells: $(ls $RESULTS_DIR/exp1/main/cells | wc -l)"
!python scripts/exp1_phenomenon.py --summary-only && head -30 $RESULTS_DIR/reports/exp1_summary_main.md

In [ ]:
# complete / re-run Exp1 on this GPU (resumable; skips finished cells)
# !python scripts/exp1_phenomenon.py --resume --batch-size 256

## Stage 1 - Exp2 (mechanism: decomposition, patching, REG sink, cross-gain preview)
Val split. 48 cells = 6 systems x L{512,2048} x {prbs, multisine} x {step, pulse}, 64 pairs each. The script prints its own time estimate for this GPU (expect roughly 1 h on L4).

In [ ]:
!python -m pytest -q tests/test_hooks.py
!python scripts/exp2_mechanism.py --quick --resume --run-name colab_quick 2>&1 | grep -E 'ESTIMATED|done|Error|Traceback'

In [ ]:
!python scripts/exp2_mechanism.py --resume 2>&1 | grep -v -E 'Loading weights|UserWarning|warnings.warn'

In [ ]:
!head -120 $RESULTS_DIR/reports/exp2_summary_main.md

## Stage 2 - Exp3 (repairs vs baselines, synthetic test) and Exp4 (real covariate + univariate)
Exp3 runs: R1 selection on val -> dither selection -> R2 gate learning -> full FT + LoRA (official `fit`) -> test evaluation (152 cells). Each stage is resumable. Exp4 reuses the Exp3 artefacts.

In [ ]:
!python -m pytest -q tests/test_repair.py
!python scripts/exp3_repair.py --quick --resume 2>&1 | grep -E 'ESTIMATED|chosen|done|Error|Traceback'
!python scripts/exp4_real_and_univariate.py --quick --resume 2>&1 | grep -E 'done|Error|Traceback'

Run `v2` (current): R2 with 32 pairs/step, lr 0.01, 400 steps, pull towards 1; fine-tuning in 64-row micro-batches x 4 (the first run `main` stopped at the start of full FT, suspected OOM). A fine-tuning failure is now logged to log.txt and the rest continues.

In [ ]:
!python scripts/exp3_repair.py --resume --run-name v2 2>&1 | grep -v -E 'Loading weights|Writing model|UserWarning|FutureWarning|warnings.warn|it/s]'
!grep -E 'FAILED|crashed|Traceback' $RESULTS_DIR/exp3/v2/log.txt; ls $RESULTS_DIR/exp3/v2

In [ ]:
!python scripts/exp4_real_and_univariate.py --resume --run-name v2 --exp3-run v2 2>&1 | grep -v -E 'Loading weights|httpx|UserWarning|warnings.warn|it/s]'

In [ ]:
!head -80 $RESULTS_DIR/reports/exp3_summary_v2.md; cat $RESULTS_DIR/reports/exp4_summary_v2.md

LoRA for run `v2` (it failed there on Colab's torchao 0.10; fixed in cell 3). This trains LoRA only (full FT and the other methods are reused), adds it to the 152 finished cells with the same samples, then re-runs Exp4 as `v2l` (Exp4 is only a few minutes).

In [ ]:
!python scripts/exp3_repair.py --resume --run-name v2 --add-methods ft_lora 2>&1 | grep -v -E 'Loading weights|Writing model|UserWarning|FutureWarning|warnings.warn|it/s]'
!grep -E 'FAILED|crashed|Traceback|added' $RESULTS_DIR/exp3/v2/log.txt | tail -5

In [ ]:
!python scripts/exp4_real_and_univariate.py --resume --run-name v2l --exp3-run v2 2>&1 | grep -v -E 'Loading weights|httpx|UserWarning|warnings.warn|it/s]'
!head -40 $RESULTS_DIR/reports/exp3_summary_v2.md; cat $RESULTS_DIR/reports/exp4_summary_v2l.md

## Later stages (cells added as they are implemented)